# <u>Hyperparameter Tuning</u>

### Prerequisites:


* <a href="Supervised ML Basis.ipynb">Check out the notebook on Supervised ML Basis</a>

* <a href="../1.Supervised Learning/Evaluation.ipynb">Check out the notebook on Evaluation</a>


    
## Topics

* [1. What is Hyperparameter Tuning?](#what)
* [2. Why Tuning Matters](#matter)
* [3. Why Tuning is Hard](#hard)
    * [3.1 Huge search spaces](search)
    * [3.2 Black-box optimization problem](box)
    * [3.3 Mixed parameter types](#types)
* [4. Untouched Test Set Principle](#test)
* [5. Nested Resampling](#resampling)
* [6. Basic Tuning Techniques](#techniques)
    * [6.1 Grid Search](#grid)
    * [6.2 Random Search](#random)
* [7. Advanced Tuning Techniques](#advanced)
    * [7.1 Evolutionary Algorithms (EAs)](#evolution)
    * [7.2 Bayesian Optimization (BO)](#bayesian)
    * [7.3 Multi-Fidelity Optimization](#fidelity)
* [8. Pipelines in Machine Learning](#pipeline)
* [9. Pipelines as DAGs and AutoML](#DAGs)
* [10. Main Challenges of AutoML / HPO](#AutoML)

In [2]:
import numpy as np # for random numbers and arrays

# Plotting
import matplotlib.pyplot as plt # for plotting
import plotly.express as px # for plotting
import plotly.graph_objects as go # for plotting

from sklearn.preprocessing import StandardScaler # standardize features

# Create Datasets
from sklearn.datasets import make_regression # create toy data for Regression
from sklearn.datasets import make_classification # create toy data for Classification
from sklearn.datasets import make_blobs # create toy data also for Classification
from sklearn.datasets import make_moons # create toy data also for Classification

from sklearn.model_selection import train_test_split # split dataset into train and test set

# Metrics for Regression
from sklearn.metrics import (
    mean_squared_error, # MSE
    mean_absolute_error, # MAE
    mean_absolute_percentage_error, # MAPE
    r2_score # R^2
)

# Metric for Classification
from sklearn.metrics import (
    accuracy_score, # Accuracy
    brier_score_loss, # Brier Score
    log_loss, # Log Loss
    confusion_matrix, # TP, FP, FN, TN
    precision_score, # Precision = PPV
    recall_score, # Recall = TPR
    roc_auc_score, # Area under ROC curve
    roc_curve, # ROC curve
    ConfusionMatrixDisplay
)

# Resampling methods
from sklearn.model_selection import (
    KFold, # K-Fold Cross Validation
    StratifiedKFold, # Stratified K-Fold Cross Validation
    LeaveOneOut, # Leave-One-Out Cross Validation
)

# ML algorithms
from sklearn.neighbors import KNeighborsClassifier # k neares neighbors
from sklearn.preprocessing import PolynomialFeatures # preprocessing for Polynomial regression
from sklearn.linear_model import LinearRegression # perform OLS
from sklearn.tree import DecisionTreeClassifier # for Classification trees
from sklearn.tree import DecisionTreeRegressor # for Regression trees
from sklearn.linear_model import Ridge # Ridge regression
from sklearn.svm import SVC # (non) linear (non)separable SVM
from sklearn.linear_model import LogisticRegression # logistic regression
from sklearn.naive_bayes import GaussianNB # Naive Bayes
from sklearn.ensemble import RandomForestClassifier # Random forest classifier

print("Setup complete")

Setup complete


<a class="anchor" id="what"></a>
# 1. What is Hyperparameter Tuning?

**A machine learning algorithm has two kinds of quantities:**
- Model parameters $\theta$: Learned automatically during training and are output of training (e.g. regression coefficients, tree split points)
- Hyperparameters $\lambda$: Input of training set before training the model and control how learning happens (e.g. tree depth, number of neighbors in k-NN, learning rate, number of trees)

**The process of finding a good hyperparameter configuration is called hyperparameter tuning or <u>hyperparameter optimization (HPO)</u>.**

### Core idea:
Hyperparameters determine model complexity and training behavior, so a poor choice can lead to:
- underfitting (model too simple)
- overfitting (model too complex)

Good tuning seeks the configuration that gives the best generalization performance on unseen data.

**Examples of Hyperparameters:**
- Maximum deoth of a Decision Tree
- Number of Trees of a Random Forest
- Minimal number of observations per leaf and splitting criterion (gini impurity, MCE)
- k in k-NN as well as the distance measure (euclidean or manhattan)
- Regularization strenght $\lambda$ in Ridge and Lasso Regression
- Polynomial degree $d$ in Polynomial Regression
- Number and maximal order of interactions for Multiple linear Regression
 

<a class="anchor" id="matter"></a>
# 2. Why Tuning Matters

**Even strong algorithms can perform badly if hyperparameters are poorly chosen.**

Example: A classification tree with depth = 4 may be
- too shallow for complex data, or
- too deep for simple data.

So we must systematically test several values and compare predictive performance.


**Goal:Find the hyperparameter vector $\lambda^*$ that minimizes estimated generalization error.**


In [3]:
help(make_blobs)

Help on function make_blobs in module sklearn.datasets._samples_generator:

make_blobs(
    n_samples=100,
    n_features=2,
    *,
    centers=None,
    cluster_std=1.0,
    center_box=(-10.0, 10.0),
    shuffle=True,
    random_state=None,
    return_centers=False
)
    Generate isotropic Gaussian blobs for clustering.

    Read more in the :ref:`User Guide <sample_generators>`.

    Parameters
    ----------
    n_samples : int or array-like, default=100
        If int, it is the total number of points equally divided among
        clusters.
        If array-like, each element of the sequence indicates
        the number of samples per cluster.

        .. versionchanged:: v0.20
            one can now pass an array-like to the ``n_samples`` parameter

    n_features : int, default=2
        The number of features for each sample.

    centers : int or array-like of shape (n_centers, n_features), default=None
        The number of centers to generate, or the fixed center locations.


In [ ]:
n = 100
X,y = make_blobs(n_samples=n,n_features=5,centers=4,cluster_std=1)

max_depths = np.arange(20)
neighbors = np.arange(20)
C_vals = np.linspace(0,100,len(neighbors))
k_fold = KFold(n_splits=4, shuffle=True, random_state=1955)

for max_depth,k,C in zip(max_depths,neighbors,C_vals):
    # ML models
    ml_models = {"Classification Tree": DecisionTreeClassifier(max_depth=max_depth,random_state=1847),
            "k-Neares Neighbors":KNeighborsClassifier(n_neighbors=k),
            "SVM":SVC(C=C)}
    for name, model in ml_models.items():
        for train_idx, test_idx in k_fold.split(X):
            pass

    


TypeError: 'KFold' object is not iterable

In [17]:
for fold, (train_idx, test_idx) in enumerate(k_fold.split(X),1):
    print(fold,train_idx,test_idx)


1 [ 0  1  4  6  7  9 10 11 12 14 15 16 17 19 20 25 27 28 29 30 31 32 35 36
 37 38 39 40 41 43 44 45 46 47 48 49 51 52 53 54 55 56 57 59 60 61 62 65
 66 67 68 69 70 71 72 73 74 75 77 79 80 81 82 83 85 86 87 88 90 91 95 96
 97 98 99] [ 2  3  5  8 13 18 21 22 23 24 26 33 34 42 50 58 63 64 76 78 84 89 92 93
 94]
2 [ 0  1  2  3  4  5  6  8  9 10 11 12 13 14 16 17 18 19 21 22 23 24 25 26
 27 29 31 32 33 34 36 38 39 41 42 43 44 47 48 50 53 54 56 57 58 60 61 62
 63 64 68 69 70 71 72 74 75 76 77 78 81 82 84 85 86 87 89 90 92 93 94 95
 96 97 99] [ 7 15 20 28 30 35 37 40 45 46 49 51 52 55 59 65 66 67 73 79 80 83 88 91
 98]
3 [ 0  1  2  3  5  6  7  8  9 12 13 15 17 18 20 21 22 23 24 26 27 28 30 32
 33 34 35 36 37 39 40 42 45 46 47 48 49 50 51 52 53 54 55 58 59 61 63 64
 65 66 67 68 70 72 73 76 77 78 79 80 81 83 84 86 87 88 89 91 92 93 94 95
 96 98 99] [ 4 10 11 14 16 19 25 29 31 38 41 43 44 56 57 60 62 69 71 74 75 82 85 90
 97]
4 [ 2  3  4  5  7  8 10 11 13 14 15 16 18 19 20 21 22 23 24 25 26 28 2

<a class="anchor" id="hard"></a>
# 3. Why Tuning is Hard

**Hyperparameter optimization is challenging for multiple reasons.**

<a class="anchor" id="search"></a>
## 3.1 Huge search spaces

If several hyperparameters each have many possible values, the number of combinations grows exponentially.

Hyperparameter search space $\Lambda$ grows exponentially.

<a class="anchor" id="box"></a>
## 3.2 Black-box optimization problem

We cannot directly compute how good a hyperparameter setting $\lambda \in \Lambda$ is (no gradients,only evaluation).

To evaluate one configuration, we must:

1. train the model,
2. validate/test it,
3. compute a performance metric.

So each evaluation is expensive.

<a class="anchor" id="types"></a>
## 3.3 Mixed parameter types

Summarize all Hyperparameters in a vector $\lambda \in \Lambda$ with possibly mixed types:

- continuous
    - e.g. minimal error improvement in a tree to accept a split
- integer
    - e.g. Neighborhood size k in k-NN
    - e.g. number of base learners (trees) in Random Forests
- categorical
    - e.g. Split criterion for Classification Trees
    - e.g. Distance measure for k-NN
- hierarchical/dependent
    - e.g. Tree depth depends on minimal leaf size we allow

This makes standard calculus-based optimization unusable.

<a class="anchor" id="test"></a>
# 4. Untouched Test Set Principle

**We must not use the same data both to tune hyperparameters and to estimate final performance.**

Otherwise performance estimates of $\widehat{\text{GE}}$ become optimistically biased.

Correct approach:

- use training/validation data to tune,
- keep a final test set untouched until the very end

<p align="center">
<img src="pics/38.png" width="600"/>
</p>

In [ ]:
#

<a class="anchor" id="resampling"></a>
# 5. Nested Resampling

To get a robust and unbiased estimate $\widehat{\text{GE}}$ with low variance, perfrom nested resampling:

Outer loop (performance estimations): 

- k-fold CV gives multiple test sets
- Averaging across folds reduces variance of the $\text{GE}$ estimate

Inner loop (hyperparameter tuning):

- For each outer training set inner m-fold CV is used to select the best $\lambda \in \Lambda$ denoted as $\lambda^*$

Instead of relying on a single split we average across many train/validation/test splits which reduces variance of estimates $\widehat{\text{GE}}$ of  $\text{GE}$

$\Rightarrow$ This mimics the untouched-test-set principle while reducing variance compared with one single holdout split.

<p align="center">
<img src="pics/39.png" width="600"/>
</p>

In [ ]:
#

<a class="anchor" id="techniques"></a>
# 6. Basic Tuning Techniques


<a class="anchor" id="grid"></a>
## 6.1 Grid Search

**Choose a finite list of candidate values for each hyperparameter and test all combinations.**

Advantages
- simple
- works for all parameter types
- easy to parallelize

Disadvantages
- combinatorial explosion
- inefficient because many irrelevant regions are tested
- requires arbitrary discretization choices

<p align="center">
<img src="pics/40.png" width="600"/>
</p>

In [ ]:
#

<a class="anchor" id="random"></a>
## 6.2 Random Search

**Instead of exhaustively testing a grid, sample configurations randomly (uniformly).**

Advantages
- simple
- can stop anytime
- often explores important dimensions better than grid search

Disadvantages
- still inefficient in high dimensions
- may waste trials in poor regions

Important takeaway:

&nbsp; Random search often better because not all hyperparameters matter equally.

<p align="center">
<img src="pics/41.png" width="600"/>
</p>

In [ ]:
#

<a class="anchor" id="advanced"></a>
# 7. Advanced Tuning Techniques

<a class="anchor" id="evolution"></a>
## 7.1 Evolutionary Algorithms (EAs)

**Idea:** EAs stochastic, population based optimization algorithm. Instead of optimizing one Hyperparameter configuration (HPC) at a time, they maintain a population of candidate solutions and improve them over generations using evolutionary principles.

**How it works in HPO:**
1. Initialization: Start with random population of HPC $\lambda^{(1)},\lambda^{(2)},\ldots,\lambda^{(\text{pop})}$
2. Selecttion: Choose best-performing candidates (parents) to "survive"
    - Common strategies: Roulette wheel, uniform selection
3. Crossover: Combine parts of two parents to form a new candidate
    - e.g. Parent A: max_depth = 5,Parent B: max_depth = 10 $\Rightarrow$ Child: max_depth = 7
4. Mutation: Randomly change some Hyper parameters (HP) of a candidate
5. Evaluation: Measure performance (e.g. accuracy, AUV,log loss) of each HPC
6. Selection: Insert new offspring into the population, possibly replacing the worst candidate


EAs is useful because they do not need gradients and can search complex spaces.

<p align="center">
<img src="pics/42.png" width="600"/>
</p>

In [ ]:
#

<a class="anchor" id="bayesian"></a>
## 7.2 Bayesian Optimization (BO)

BO treats HPO as a black-box function optimization problem:
$$
f(\lambda) = \widehat{\text{GE}}(\mathcal{I},\mathcal{J},\rho,\lambda)
$$
- $\mathcal{I}$ is the learner/inducer which takes data as input and searches through possible machine learning models and finds the model that fits the data best according to some risk measure $\rho$
- $\mathcal{J}$ is the resampling splits

Problem: $f$ is expensive to evaluate (requires training & validation)
Solution: BO builds a cheap surrogate model of $f$ and uses it to giude which HPC to try

**How it works:**
1. Surrogate model: Approximate relationship between HP $\lambda$ and performance $c(\lambda)$
  - Often Gaussian Process (GPs) for continuous parameters
  - Surrogate provides mean prediction $\hat{c}(\lambda)$ and uncertainty estimate $\hat{\sigma}(\lambda)$ 

2. Acquisition function: Decides which HP to try next by balancing:
  - Exploration: Sample where uncertainty is high
  - Exploitation: Sample near promising (low error/high accuracy) areas
  - Common Acquisition functions
    - Expected Improvement (EI)
    - Probability of Improvement (PI)
    - Lower Confidence Bound (LCB)

3. Optimization Loop
  - Select new candidates by maximizing aquisition function
  - Evaluate them with CV or resampling
  - Update surrogate model with new data
  - Repeat until budget is exhausted


**Nutshell:** Instead of blindly trying points, Bayesian Optimization:

1. builds a surrogate model that predicts performance across the search space,
2. uses an acquisition function to decide where to test next,
3. balances:
    - exploration = trying uncertain regions
    - exploitation = refining promising regions.

This allows it to focus evaluations on "interesting" areas and usually outperform random search when evaluations are expensive.

<div style="display: flex; justify-content: center; gap: 10px;">
  <img src="pics/43.png" width="550"/>
  <img src="pics/44.png" width="550"/>
</div>

In [ ]:
#

<a class="anchor" id="bayesian"></a>
## 7.3 Multi-Fidelity Optimization

**Motivation:** Training ML models for every HPC to full budget (e.g. all epochs, full dataset) is too expensive. But often we can tell early on if a HPC is poor. Hyperband (HB) solves this by allocating variable budgets and using successive halving to drop bad HPC quickly.

**How it works:**
##### 1. Budget definition
- A budget is chosen, $\overbrace{\text{e.g. number of epochs, dataset, fraction or iterations}}^{\text{fidelity HP} (\lambda_{\text{fid}})}$

##### 2. Successive Halving (SH)
- Start with many HPCs, train them with a small budget
- Keep only the top fraction (e.g. $\frac{1}{\eta}$ with $\eta=2,3$ or 4)
- Double or triple the budget for survivers
- Repeat until one HPC remains

##### 3. Hyperband = many SH run with different starting budgets
- To avoid killing off good HPCs too early (a weakness of SH), Hyperband restarts SH multiple times with different trade-offs:
    - Some run with many HPCs + tiny initial budget
    - Others with fewer HPCs + larger initial budget
$\Rightarrow$ Ensures good coverage of trade-offs between breadth (exploring many HPCs) and depth (fully training fewer HPCs)



In [ ]:
#

<a class="anchor" id="pipeline"></a>
# 8. Pipelines in Machine Learning

**Hyperparameter tuning should not only tune the learner, but the entire model-building process.**

A pipeline is a structured sequence of steps the data goes thourh in ML workflows. It is a connected chain of operations:

- preprocessing,
- feature engineering,
- model fitting,
- predicting,
- (postprocessing).

Benefits:

- automation,
- reproducibility,
- fewer human errors,
- prevents data leakage.

Every pipeline component may have its own hyperparameters. Therefore tuning the pipeline means tuning the joint hyperparameter space of all steps.

**Types of pipelines**
1. Sequential pipeline(linear flow)
- Raw data $\rightarrow$ Preprocessing $\rightarrow$ Model $\rightarrow$ Prediction
2. DAG (Directed Acyclic Graph) pipeline
- Allows branching and combining 

<p align="center">
<img src="pics/45.png" width="600"/>
</p>

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ("scaler", StandardScaler()), 
    ("rf", RandomForestClassifier(n_estimators=100))
])

pipeline.fit(X, y)
pipeline.predict(X)
```

In [ ]:
#

<a class="anchor" id="DAGs"></a>
# 9. Pipelines as DAGs and AutoML

Pipelines can become more flexible by representing them as directed acyclic graphs (DAGs):

- alternative preprocessing branches,
- different learner choices,
- ensembles,
- operator selection.

This creates a hierarchical and conditional hyperparameter space, which is the foundation of AutoML:

> AutoML = automatically searching over preprocessing + model type + hyperparameters.

**Instead of manually choosing the steps of a pipeline, AutoML systems search automatically for the best-performing pipeline configuration**

Key insight:

&nbsp;&nbsp; AutoML succeeds when:

- the pipeline graph is expressive enough, and
- the tuner is efficient enough

<p align="center">
<img src="pics/46.png" width="600"/>
</p>

<a class="anchor" id="AutoML"></a>
# 10. Main Challenges of AutoML / HPO

**Unresolved issues:**

- computational cost,
- integrating expert prior knowledge into AutoML,
- transferring experience between tasks,
- balancing multiple objectives (accuracy, interpretability, runtime),
- reducing black-box behavior for better trust